In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw").exists():
    for candidate in [PROJECT_ROOT.parent, PROJECT_ROOT / "notebook"]:
        if (candidate / "data" / "raw").exists():
            PROJECT_ROOT = candidate
            break
DB_PATH = PROJECT_ROOT / "data" / "raw" / "airtel.db"


In [ ]:
conn = sqlite3.connect(DB_PATH)
query = """
SELECT circle_name
FROM circles
WHERE circle_name = 'circles';
"""
df = pd.read_sql_query(query, conn)
display(df)
conn.close()

In [ ]:
conn = sqlite3.connect(DB_PATH)
query = """
SELECT
    COUNT(DISTINCT region) AS distinct_regions,
    SUM(region IS NULL) AS nulls,
    SUM(TRIM(region) = '') AS blanks,
    ROUND(
        100.0 * AVG(
            region IS NULL OR TRIM(region) = ''
        ), 2
    ) AS pct_missing,
    MIN(region) AS smallest,
    MAX(region) AS largest
FROM circles;
"""
df = pd.read_sql_query(query, conn)
display(df)
conn.close()

In [ ]:
conn = sqlite3.connect(DB_PATH)
query = """
SELECT region, COUNT(*) AS n
FROM circles
GROUP BY region
ORDER BY n DESC;
"""
df = pd.read_sql_query(query, conn)
display(df)
conn.close()

In [ ]:
conn = sqlite3.connect(DB_PATH)
query = """
SELECT
    txn_type,
    COUNT(*) AS payments,
    COUNT(DISTINCT customer_id) AS subs,
    ROUND(AVG(amount_inr), 2) AS avg_inr
FROM recharges
GROUP BY txn_type;
"""

df = pd.read_sql_query(query, conn)
display(df)
conn.close()

In [ ]:
conn = sqlite3.connect(DB_PATH)
query1 = """
SELECT COUNT(*) AS rows_,
       COUNT(DISTINCT customer_id) AS distinct_
FROM customers;
"""
df1 = pd.read_sql_query(query1, conn)
display(df1)

In [ ]:

conn = sqlite3.connect(DB_PATH)

# SQL query to count rows in all tables
query = """
SELECT 'circles' AS tbl, COUNT(*) AS rows_
FROM circles

UNION ALL
SELECT 'plans', COUNT(*)
FROM plans

UNION ALL
SELECT 'customers', COUNT(*)
FROM customers

UNION ALL
SELECT 'usage_daily', COUNT(*)
FROM usage_daily

UNION ALL
SELECT 'recharges', COUNT(*)
FROM recharges

UNION ALL
SELECT 'complaints', COUNT(*)
FROM complaints;
"""

# Execute and display results
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:
conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
SELECT
    COUNT(DISTINCT region) AS distinct_,
    SUM(region IS NULL) AS nulls_,
    SUM(TRIM(region) = '') AS blanks_,
    ROUND(
        100.0 * AVG(
            region IS NULL OR TRIM(region) = ''
        ), 2
    ) AS pct_missing,
    MIN(region) AS smallest,
    MAX(region) AS largest
FROM circles;
"""

# Execute the query and display the result
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# 1. INNER JOIN: counts matching customer-usage rows
query_inner = """
SELECT COUNT(*) AS inner_join_rows
FROM customers AS c
JOIN usage_daily AS u
    ON u.customer_id = c.customer_id;
"""

df_inner = pd.read_sql_query(query_inner, conn)
display(df_inner)

# 2. LEFT JOIN: keeps all customers, including those without usage
query_left = """
SELECT COUNT(*) AS left_join_rows
FROM customers AS c
LEFT JOIN usage_daily AS u
    ON u.customer_id = c.customer_id;
"""

df_left = pd.read_sql_query(query_left, conn)
display(df_left)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# 1. Check customer churn status and churn date
query_customers = """
SELECT
    SUM(is_churned = 1 AND churn_date IS NOT NULL)
        AS left_dated,
    SUM(is_churned = 0 AND churn_date IS NULL)
        AS stayed_blank
FROM customers;
"""

df_customers = pd.read_sql_query(query_customers, conn)
display(df_customers)

# 2. Check unresolved complaints with missing resolution days
query_complaints = """
SELECT
    SUM(is_resolved = 0 AND resolution_days IS NULL)
        AS open_blank
FROM complaints;
"""

df_complaints = pd.read_sql_query(query_complaints, conn)
display(df_complaints)

conn.close()


In [ ]:
conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
SELECT
    r.txn_type,
    c.connection_type,
    COUNT(DISTINCT r.customer_id) AS subs
FROM recharges AS r
JOIN customers AS c
    ON c.customer_id = r.customer_id
GROUP BY r.txn_type, c.connection_type;
"""

# Execute the query and display results
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
SELECT
    current_plan_id,
    COUNT(DISTINCT arpu_inr) AS values_,
    COUNT(*) AS subs
FROM customers
GROUP BY current_plan_id;
"""

# Execute and display results
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
WITH r AS (
    SELECT
        customer_id,
        SUM(amount_inr) / 6.0 AS actual
    FROM recharges
    WHERE recharge_date BETWEEN
          '2025-01-01' AND '2025-06-30'
    GROUP BY customer_id
)
SELECT
    ROUND(AVG(c.arpu_inr), 2) AS stated,
    ROUND(AVG(COALESCE(r.actual, 0)), 2) AS actual,
    SUM(r.customer_id IS NULL) AS paid_none
FROM customers AS c
LEFT JOIN r
    ON r.customer_id = c.customer_id;
"""

# Execute and display the result
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
SELECT
    ROUND(
        100.0 * SUM(r.plan_id = c.current_plan_id)
        / COUNT(*),
        2
    ) AS pct_matching
FROM recharges AS r
JOIN customers AS c
    ON c.customer_id = r.customer_id;
"""

# Execute the query and display the result
df = pd.read_sql_query(query, conn)
display(df)

conn.close()


In [ ]:

conn = sqlite3.connect(DB_PATH)

# SQL query
query = """
WITH l AS (
    SELECT
        customer_id,
        MAX(usage_date) AS last_use
    FROM usage_daily
    GROUP BY customer_id
)
SELECT
    c.is_churned,
    COUNT(*) AS subs,
    ROUND(
        AVG(
            julianday('2025-06-30') - julianday(l.last_use)
        ), 2
    ) AS avg_quiet,
    MIN(
        julianday('2025-06-30') - julianday(l.last_use)
    ) AS min_,
    MAX(
        julianday('2025-06-30') - julianday(l.last_use)
    ) AS max_
FROM customers AS c
JOIN l
    ON l.customer_id = c.customer_id
GROUP BY c.is_churned;
"""

# Execute query and display results
df = pd.read_sql_query(query, conn)
display(df)

conn.close()
